# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. We'll load data using the Croissant schema and inspect the available record sets, fields, and their `@id`s, following best practices for FAIR data exploration.

### Dataset Source
The dataset is described with a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records with `mlcroissant`, and display basic information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata
print('Dataset:', dataset.metadata.name)
print('Version:', dataset.metadata.version)
print('Identifier:', dataset.metadata.identifier)
print('Citation:', getattr(dataset.metadata, 'citeAs', None))
print('Description:', dataset.metadata.description)
print('Authors:', getattr(dataset.metadata, 'author', None))

## 2. Data Overview
Let's review available record sets, their fields, and `@id` values as defined by the Croissant schema. 

Typically these represent tables or collections of structured records in the dataset.

In [ ]:
# List all RecordSet objects
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s)\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    # List its fields
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field name: {getattr(field, 'name', None)} | @id: {field.id} | Data type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
We'll load the main record set(s) into pandas DataFrame(s) for exploration.

> All fields, record sets, and data columns are always referenced by their Croissant `@id` values for proper reproducibility and disambiguation.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Available record set @id values:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")
    print("Columns:", df.columns.tolist())
    print(df.head(2), "\n")

## 4. Exploratory Data Analysis (EDA)
Let's process and explore numeric fields in the main record set. We'll filter rows, normalize, and group by relevant categorical fields. Please substitute the relevant `@id` values from above as appropriate per your analysis goal.

In [ ]:
# Pick the first available record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Working with record set @id: {main_record_set_id}")
else:
    raise RuntimeError("No record sets found in the dataset.")

# Identify a numeric field (column) for demonstration
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    raise RuntimeError("No numeric fields found in the record set data.")

print(f"Numeric field chosen for analysis: {numeric_field_id}")

# Simple threshold filtering
if df[numeric_field_id].dtype.kind in 'biufc':
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered entries where {numeric_field_id} > {threshold:.2f}")
    print(filtered_df.head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by a categorical field
    categorical_candidates = [c for c in df.columns if pd.api.types.is_object_dtype(df[c])]
    if categorical_candidates:
        group_field_id = categorical_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric columns suitable for EDA.')

## 5. Visualization
Now let's visualize the distribution of our selected numeric field, and the effect of grouping by a categorical field. If matplotlib isn't installed, you may add a cell to install it first.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of numeric field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping was performed:
if 'group_field_id' in locals():
    plt.figure(figsize=(12,5))
    order = grouped_df.sort_values(numeric_field_id, ascending=False)[group_field_id].values
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df, order=order)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and perform basic exploration of a FAIR-compliant Croissant dataset using the `mlcroissant` Python library. All references to data tables, fields, and columns were made via their schema `@id` values for rigorous, portable data science. Continue your analysis by repeating these steps for other record sets or fields of interest.

For further details, please refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the original dataset materials from [SEN Science](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).